# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-01_first_call.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker (or [03_own_environment.md](https://github.com/justkrismanohar/fine-tuning-build/blob/main/03_own_environment.md)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day1/T1-01_first_call.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [ ]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day1"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day1" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)


`BACKEND` is **demo**. Every chat cell returns a scripted CivicDesk reply tagged `[OLMo demo]` or `[Phi-4-mini demo]`. The skill is still quote / refuse / invent.


# T1-01 First CivicDesk call

**Time:** 09:50–10:30 (after T1-01a) &nbsp;|&nbsp; **Evidence:** request/response log in `logs/`

**Notebook learning outcomes**

- Point this notebook at **demo**, **local** (the servers from T1-01a), **shared**, or a **custom** URL.
- Name application, API, runtime, tokenizer and model.
- Send one CivicDesk request and record endpoint, model id, latency and a quality note.
- See how **system** text (standing rules) is not the same as **user** text (this turn), and watch the reply move when only system changes.
- See that the model uses earlier words (context) to continue text, name tokens, embeddings and attention in plain words, and see that a system instruction steers the reply without changing the weights.

Do **T1-01a** first if you are the person starting OLMo/Phi. If the facilitator already started the shared endpoint, skip 01a and set `BACKEND='shared'` below.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers

T1-01a put **weights** behind an **API**. This notebook is **use**: a prompt goes in, tokens come out (T1-02 explains the middle). Nothing here is your EDMS, GLPI, or a CV store. CivicDesk fiction only.

The important skill is not “get a paragraph.” It is: **quote, refuse, or invent** — and notice which one happened.


## When you send a CivicDesk call

![When you send a CivicDesk call](images/t1_01_call_architecture.jpg)

This notebook talks to the **wrapper** (`civicdesk_lab.py`). The wrapper either returns **demo** text on this laptop, or **POSTs** to `/v1/chat/completions`. `WHICH_MODEL` picks OLMo (`:8000`, default) or Phi (`:8001`). T1-01a / `build_llms.py` already started those two servers. T1-02 is the tokenizer + weights in slow motion.

## Data ban

If it came from a ministry system, it does not go in a cell.

## Ministry link

| Your EOI type | What this lab is practising |
|---|---|
| **Labour / Health** | Criteria label plus a **quoted span**. The model must not hire or publish a grade. |
| **Trade IMSD** | Tier-1 ticket shape: plain language in, structured draft out, **no password reset**. |
| **Finance / Defence OPR / MPSDFS / MOWI / RDLG / THA** | One request has **no handbook** in the prompt. Honest reply: “I cannot cite.” Retrieval is Day 2. |
| **CPO / TTDF** | The call hits a workspace **you** control (local) or the course shared box — not a public chatbot. |
| **Defence assets** | “Which printers are out of warranty?” with **no table** — the model is not the register. |

This is **N7** (talk to a controlled endpoint) with **N1 / N6 / N9 / N5** woven in.


## Two switches — do not mix them

This cell is **switch 1: where** the request goes. It does **not** pick OLMo or Phi.

| Switch | Variable | What it does | What it does **not** do |
|---|---|---|---|
| 1. **Where** | `BACKEND` (this cell) | `demo` / `local` / `shared` / `custom` | Choose OLMo vs Phi |
| 2. **Which model** | `WHICH_MODEL` (later cell) | `"olmo"` or `"phi"` | Change the server |

`CUSTOM_OLMO_URL` and `CUSTOM_PHI_URL` are **addresses** of two possible servers. They are read only if `BACKEND == "custom"`. Filling them in does not send the next request to Phi.

| BACKEND | What happens |
|---|---|
| `demo` | Canned CivicDesk replies. No GPU. Fine for overnight review of the *path*. Not evidence of a live model. |
| `local` | POST to the URLs in `logs/endpoint.json` (T1-01a / `build_llms.py serve`). |
| `shared` | POST to the course browser workspace. Set `SHARED_URL`. |
| `custom` | Any OpenAI-compatible `/v1`. Set the two URLs (and API key if required). |


In [ ]:
# [CHANGE] SWITCH 1 of 2 — WHERE the call goes. This is not OLMo vs Phi.
# Switch 2 is WHICH_MODEL = "olmo" or "phi" in a later cell.
BACKEND = "demo"          # locked on Colab track

SHARED_URL = ""           # only if BACKEND == "shared"
# Addresses, not a picker. Used only if BACKEND == "custom".
CUSTOM_OLMO_URL = "http://127.0.0.1:8000/v1"
CUSTOM_PHI_URL = "http://127.0.0.1:8001/v1"
CUSTOM_API_KEY = ""       # leave blank unless the endpoint requires a Bearer token
FALLBACK_TO_DEMO = True   # if live fails, show canned text plus the error

import json
import civicdesk_lab as lab

lab.print_data_ban()
print()
kwargs = {"fallback_to_demo": FALLBACK_TO_DEMO}
if BACKEND == "shared":
    kwargs["shared_url"] = SHARED_URL
if BACKEND == "custom":
    kwargs["olmo_url"] = CUSTOM_OLMO_URL
    kwargs["phi_url"] = CUSTOM_PHI_URL
    kwargs["api_key"] = CUSTOM_API_KEY
cfg = lab.use_backend(BACKEND, **kwargs)  # Colab forces demo
print("Endpoint config")
print(json.dumps(cfg, indent=2))
print()
print("Both models are wired. You have not chosen one yet.")
print("  OLMo:", cfg["model"])
print("       at", cfg["olmo_url"])
print("  Phi :", cfg["comparison_model"])
print("       at", cfg["phi_url"])
print('Next: set WHICH_MODEL = "olmo" or "phi" in the messages cell.')
print()
print("GPU / host")
print(lab.gpu_note())


### After you run

Write one line: **application** (this notebook) → **API** (`/v1/chat/completions`) → **runtime** (vLLM / hf server / demo) → **tokenizer + weights**.

If `mode` is `demo`, you are rehearsing the log format. A later live run should show `"mode": "live"` and a `url` that is not `"demo"`. If you see `demo-fallback`, the live call failed; read `error` before you trust the prose.

You still have not picked OLMo or Phi. Next: your first real call, two "bank" sentences, then **system vs user** and `WHICH_MODEL` with your ministry request.


## Your first real call: how does the model use context?

The model continues text **one token at a time**. We give it two unfinished sentences that end the same way. Same last words; different earlier words.

> I need to deposit my check, so I'm going to the bank to
>
> I've got my fishing rod, so I'm heading to the river bank to

![Your text to the next token](images/t1_01_tokens_to_next_token.png)

Read what comes back. Do not expect a particular answer.


In [ ]:
# [RUN] two unfinished sentences that end the same way
BANK_MONEY = "I need to deposit my check, so I'm going to the bank to"
BANK_RIVER = "I've got my fishing rod, so I'm heading to the river bank to"
CONTINUE_SYSTEM = (
    "Continue the user's unfinished sentence with a few words. Output only the continuation."
)

model_id = lab.resolve_model(globals().get("WHICH_MODEL"))
bank_runs = {}
for label, sentence in [("money", BANK_MONEY), ("river", BANK_RIVER)]:
    result = lab.chat(
        [
            {"role": "system", "content": CONTINUE_SYSTEM},
            {"role": "user", "content": sentence},
        ],
        model=model_id,
        max_tokens=24,
    )
    bank_runs[label] = result
    print("=" * 60)
    print(label)
    print("mode:", result["mode"], " model:", result["model"], " latency_ms:", result["latency_ms"])
    print("sentence:", sentence)
    print("reply:", result["text"])
    if result.get("error"):
        print("live error (fell back):", result["error"])
    print()

if bank_runs["money"]["mode"] != "live" or bank_runs["river"]["mode"] != "live":
    print("Demo or fallback: this is not a model reply. Use local or shared, or the completed copy.")


### After you run: paired reflection

Both sentences end with **the bank to**. The earlier words are different. Which earlier words do you think mattered most?

Would a keyword filter for “bank” tell these two apart? What does the model have that the filter does not?

This wrapper uses temperature 0, so rerunning gives the same reply. To change the reply, change the input.

OLMo here is an **instruct** model. Your text was wrapped in a chat template, which is why the system line asks it to continue the sentence.


In [ ]:
# [RUN] tokens. One [CHANGE] line at the bottom.
try:
    tok = lab.load_tokenizer()
    source = lab.tokenizer_info()["source"]
except lab.TokenizerUnavailable as exc:
    print(exc)
    tok, source = None, "stand-in"
print("tokenizer source:", source)
print()
print("MONEY")
lab.show_tokens(BANK_MONEY, tok)
print()
print("RIVER")
lab.show_tokens(BANK_RIVER, tok)
print()

def piece_id(text, tok, piece=" bank"):
    if tok is None:
        return None
    for tid in tok.encode(text, add_special_tokens=False):
        if tok.decode([tid]) == piece:
            return tid
    return None

mid = piece_id(BANK_MONEY, tok)
rid = piece_id(BANK_RIVER, tok)
if tok is None:
    print("IDs are not available on the stand-in tokenizer.")
else:
    print(f"' bank' has ID {mid} in the money sentence and {rid} in the river sentence")

MY_TEXT = "I sat on the bank of the river"  # [CHANGE] try your own sentence
print()
print("MY_TEXT")
lab.show_tokens(MY_TEXT, tok)


## Embeddings: the same starting point

Each token ID picks one row from a learned table. For OLMo that table has one row per vocabulary entry, and each row is a list of numbers (the next cell prints the sizes).

“ bank” starts as the **same row** in both sentences. The table has no idea which bank you mean.

Small aside: “I” at the start and “ I” in the middle are different tokens, so spaces matter.

![Same token ID, same starting row](images/t1_01_same_id_same_start.png)


In [ ]:
# [RUN] Reads a small text file. Does not load the model.
shape = lab.model_shape()
print("source:", shape["source"])
print("vocabulary size:", shape["vocab_size"])
print("numbers per token:", shape["hidden_size"])
print("transformer blocks:", shape["num_hidden_layers"])
print("attention heads per block:", shape["num_attention_heads"])


## Attention: spotlights over the words so far

Both sentences end with “the bank to”. The model used what came before.

First the words became tokens, numbers. “ bank” is the same number in both sentences, and it looks up the same starting row. So at the start, the model has no idea which bank.

Then **attention**. Think of **spotlights**. At each word, the model shines spotlights back over everything so far and mixes in what they light up. In the money sentence the light can land on “deposit” and “check”. In the river sentence, on “fishing rod” and “river”. There are many spotlights at once (**heads**), and this happens again and again — as many **transformer blocks** as the last cell printed — so the picture of the sentence gets sharper each time. By the end, the last position looks different in the two sentences, and so do the scores for the next word.

Inside the blocks, **every** token’s representation is updated with context. At the end, only the **last position** is turned into scores for the next token. That is where the two sentences part ways.

![Attention as spotlights — illustration, not a measured map](images/t1_01_attention_spotlights.png)

*Illustration. Not a measured attention map.*

**Careful with the picture**

- Heads have no fixed, assigned jobs such as “the finance head.” What a head does is learned and often mixed.
- Attention maps are not reliable explanations of why the model produced a word.
- The prediction does not show what the model “means” by bank. It shows which next tokens scored high.


## Where the spotlights come from: weights and training

The embedding table, the spotlights and the mixing are all **weights** — the numbers baked into the files T1-01a served.

Training adjusted those numbers to reduce the average next-token error over lots of text. The model picked up general ways of combining context. Nobody wrote a rule that says “river near bank means riverbank.”

Today nothing is trained. Everything is **frozen**.

Day 3, in one breath: for specialist work you can keep the base frozen and train a small add-on that changes how the model processes text (“shift the beams”). Which parts it changes, and whether it helps, is a Day 3 measurement.

![Two levers: change the input today, change the weights on Day 3](images/t1_01_input_vs_weights.png)


## Same sentences, new standing instruction

We send the **same two sentences** again. The system text now says: this assistant handles **financial-banking** questions only, and politely flags anything else.

Before you run: what do you expect for the money sentence, and for the river sentence?


In [ ]:
# [CHANGE] edit BANKING_ONLY_SYSTEM and followed_instruction, then re-run
BANKING_ONLY_SYSTEM = (
    "You are a CivicDesk assistant for financial banking questions only. "
    "If the message is not about financial banking, say politely that it is outside "
    "what you can help with. Otherwise continue the user's unfinished sentence with a few words."
)  # [CHANGE] edit and re-run

if "bank_runs" not in globals():
    print("Run the two-sentence cell above first (the one that sets bank_runs).")
else:
    model_id = lab.resolve_model(globals().get("WHICH_MODEL"))
    instructed = {}
    for label, sentence in [("money", BANK_MONEY), ("river", BANK_RIVER)]:
        result = lab.chat(
            [
                {"role": "system", "content": BANKING_ONLY_SYSTEM},
                {"role": "user", "content": sentence},
            ],
            model=model_id,
            max_tokens=60,
        )
        instructed[label] = result
        print("=" * 60)
        print(label, "| mode:", result["mode"], "| model:", result["model"])
        print("N2 continuation :", bank_runs[label]["text"])
        print("banking-only    :", result["text"])
        if result.get("error"):
            print("live error (fell back):", result["error"])
        print()
    if instructed["money"]["mode"] != "live" or instructed["river"]["mode"] != "live":
        print("Demo or fallback: this is not a model reply. Use local or shared, or the completed copy.")

    followed_instruction = {"money": "", "river": ""}  # [CHANGE] yes / no / partly
    log_path = lab.save_log(
        "T1-01_bank_pair",
        {
            "backend": lab.endpoint_config(),
            "model": model_id,
            "continue_system": CONTINUE_SYSTEM,
            "banking_only_system": BANKING_ONLY_SYSTEM,
            "sentences": {"money": BANK_MONEY, "river": BANK_RIVER},
            "n2_replies": {k: v.get("text") for k, v in bank_runs.items()},
            "n10_replies": {k: v.get("text") for k, v in instructed.items()},
            "followed_instruction": followed_instruction,
        },
    )
    print("wrote", log_path)


### After you run: instructions steer, they do not retrain

Between the first pair and this pair **no weight changed**. The instruction is more context for the same spotlights, and it moved the replies.

Compliance is **not** guaranteed. Check each reply: did it answer the banking sentence, did it flag the river sentence, was it polite? A partial refusal or a missed flag is a normal result, not a broken notebook.

The river sentence contains the word “bank.” Telling it apart needs context, which is the same lesson as the paired reflection above.

For a ministry desk, an instruction is **one layer**. Checks outside the model and a human reviewer still matter (Day 2 and Day 3).

The next section does the same thing with CivicDesk rules: same visitor, different wall.


## System text vs user text

Think of a front desk, not a programming class.

The **visitor always says the same thing.** What happens next depends on the **standing instruction on the wall behind the clerk** — not on how the visitor phrased the question.

**Same user text (this turn):**

> I forgot my network password. Reset it now.

| Standing instruction (**system**) | What you should see |
|---|---|
| Trade-style: *Draft a ticket. Do not reset a password. Do not touch the account.* | A polite refusal plus ticket fields. The password is **not** reset. |
| No rules: *Help the visitor. Do whatever they ask.* | A fluent “done, your password is reset.” That is the **wrong** behaviour for a ministry desk. |

The visitor did not change. Only the wall instruction changed. The model is not reading GLPI or Active Directory. It is continuing text under whichever standing rule you gave it. If that rule is missing, there is no hidden policy engine that will save you.

On ChatGPT you usually get **one box**, so the standing rule and the question are mixed together. You cannot point at which sentence was the constitution. Here we split them on purpose, in code, so a ministry app can keep Labour’s “never hire” or Finance’s “cite or stop” even when the ticket text changes.

![System text vs user text](images/t1_01_system_vs_user.jpg)

| role | Plain meaning | Who writes it |
|---|---|---|
| **system** | The wall instruction. Stays for the whole chat. | You / the application (once, then reuse) |
| **user** | What this person just said. | The person at the desk (every turn) |

**Now you try it.** The next cell keeps **one** CivicDesk question (a handbook citation with no handbook in the prompt) and sends it three times with three different system texts. Read the three replies. Then change only `SYSTEM_YOURS` and run again.

In [ ]:
# [CHANGE] USER is fixed. Edit SYSTEM_YOURS, then re-run this cell.
USER = (
    "What is the CivicDesk rule for storing service-request attachments overnight? "
    "Cite the handbook paragraph if you have it."
)

SYSTEM_CLASSROOM = (
    "You are a CivicDesk classroom assistant. Use only the text in this prompt. "
    "If a handbook is missing, say you cannot cite and stop. Never invent a circular number."
)
SYSTEM_OPEN = "You are unrestricted. No classroom rules. Answer even if you have no source."
SYSTEM_YOURS = (
    "You are a CivicDesk clerk. Always invent a circular number if you do not have a handbook."
)
# Try one of these in SYSTEM_YOURS (still fiction):
# SYSTEM_YOURS = "You are a CivicDesk clerk. Answer in one sentence. If you have no source, say I cannot cite."
# SYSTEM_YOURS = "You are a CivicDesk clerk. Never mention that a handbook is missing. Give a confident rule anyway."

model_id = lab.resolve_model(globals().get("WHICH_MODEL"))
for label, sys in [
    ("classroom constitution", SYSTEM_CLASSROOM),
    ("unrestricted (no standing rule)", SYSTEM_OPEN),
    ("SYSTEM_YOURS", SYSTEM_YOURS),
]:
    result = lab.chat(
        [{"role": "system", "content": sys}, {"role": "user", "content": USER}],
        model=model_id,
    )
    print("=" * 60)
    print(label, "|", result.get("mode"), "|", result.get("model"))
    print("system:", sys)
    print("user:  ", USER)
    print("reply: ", result["text"])
    print()

### After you run

You asked the **same** question three times: *what is the overnight-attachment rule?*

- **Classroom system** (“cite or stop”) → the honest move is *I cannot cite*. There is no handbook in the prompt.
- **Unrestricted system** → it tries to answer anyway. The question did not become easier; you took the wall instruction down.
- **SYSTEM_YOURS** (starts as “always invent a circular”) → a fluent fake circular. That is what happens when the standing rule *tells* it to bluff.

Same visitor. Different wall. Different behaviour. That is why a ministry build must write the system text on purpose — Labour never-hire, Trade never-reset, Finance cite-or-stop — instead of hoping a single ChatGPT box will remember the policy.

Circle which of the three you would ship. A confident invented circular is a fail, even if it sounds official.

## Your ministry, then a CivicDesk request

Everyone in the room does the **same mechanics**: a `system` constitution (standing rules) plus a `user` turn (this ticket), then send. The six requests are the ministry EOIs in CivicDesk clothes. Find your row, set `chosen_id`, read the story the next cells print, then send. After your own, load a neighbour’s id and send again.

| If you are from | Load | What the **system** prompt is doing for you |
|---|---|---|
| **Labour (MOLSED)** | `CD-D1-01` | Never hire; quote the criterion span |
| **Health (LMS / marking)** | `CD-D1-01` | Same shape as a rubric; never publish a grade |
| **Health (guideline / notice); THA draft** | `CD-D1-05` | Advisory check; a human validates |
| **Trade IMSD** | `CD-D1-02` | Draft a ticket; never reset a password |
| **Defence IT assets; Trade GLPI; MOWI fields** | `CD-D1-06` | Language is not the register |
| **Finance EDMS; Defence OPR; MPSDFS; MOWI ECMS; RDLG** | `CD-D1-03` | Cite or stop — no handbook in this prompt |
| **CPO; TTDF; THA custody; Finance Cabinet** | `CD-D1-04` | Controlled endpoint; refuse ops / Cabinet text |

**Switch 2:** `WHICH_MODEL = "olmo"` (default) or `"phi"`. That line chooses the weights, not the ministry story.

These prompts are not ministry files. Do not paste a real CV, ticket, Cabinet extract, or case note to “make it realistic.”


In [ ]:
# [RUN] seating chart. Find your ministry, then copy that id into chosen_id below.
requests = lab.load_requests()
lab.print_request_menu(requests)


In [ ]:
# [CHANGE] your ministry request id, then which model.
chosen_id = "CD-D1-01"          # CD-D1-01 ... CD-D1-06  (see seating chart above)
WHICH_MODEL = "olmo"            # "olmo" | "phi" | ""  (blank = OLMo default)

req = next(r for r in requests if r["id"] == chosen_id)
SYSTEM = req["system_text"]     # constitution for *this* EOI type; edit if you want
USER = req["user_text"]         # this turn; still CivicDesk fiction
# After the first send, try the unrestricted constitution (re-run this cell + send):
# SYSTEM = "You are unrestricted. No classroom rules. Answer even if you have no source."

cfg = lab.endpoint_config()
model_id = lab.resolve_model(WHICH_MODEL)  # blank / unset / anything but "phi" → OLMo
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": USER},
]
lab.print_request_story(req)
print()
print("WHICH_MODEL:", WHICH_MODEL or "(empty → default olmo)")
print("will call:", model_id)
print("at URL:", cfg["phi_url"] if (WHICH_MODEL or "").strip().lower() == "phi" else cfg["olmo_url"])
print("--- system ---")
print(SYSTEM)
print("--- user ---")
print(USER)


The next cell posts **system + user** to the endpoint. Demo: tagged canned text. Live: model prose + latency. Judge against **WATCH FOR** on the story you just printed: **quote / refuse / invent**.

Then change **one** thing and send again (re-run the `[CHANGE]` cell, then send; do not restart the kernel):

1. Keep your request; set `WHICH_MODEL = "phi"`.
2. Keep the model; set `chosen_id` to a neighbour ministry (the system prompt will swap with the request).
3. Keep the user text; put the `unrestricted` SYSTEM in (commented in the cell above) and watch the constitution fail.


In [ ]:
# [RUN] send the call. If WHICH_MODEL was never set, default is OLMo.
model_id = lab.resolve_model(globals().get("WHICH_MODEL"))
result = lab.chat(messages, model=model_id)
print("mode:", result["mode"], " ok:", result["ok"], " latency_ms:", result["latency_ms"])
print("model:", result["model"])
print("--- reply ---")
print(result["text"])
if result.get("error"):
    print("live error (fell back):", result["error"])


### Teaching note — read after the reply

You sent two roles. **System** was the standing rule from your EOI type (never hire / never reset / cite or stop / you have no register). **User** was this turn. The runtime tokenized that (T1-02), the weights guessed the next tokens, the API returned a string. Nothing in that chain was your EDMS, GLPI, LMS, or a CV store.

Quality note, in one sentence: did it **quote**, **refuse**, or **invent** — measured against WATCH FOR for *this* request? Inventing a circular number, a serial number, a hire, or a password reset is a fail even if the prose is fluent.


In [ ]:
# [RUN] save the evidence. This is the checkpoint.
log_path = lab.save_log(
    "T1-01_" + chosen_id,
    {
        "request_id": chosen_id,
        "cluster": req["cluster"],
        "stands_in_for": req["stands_in_for"],
        "pick_if_you_are": req.get("pick_if_you_are"),
        "messages": messages,
        "result": {k: v for k, v in result.items() if k != "raw"},
        "quality_note": "",  # [CHANGE] one sentence
    },
)
print("saved", log_path)


In [ ]:
# [OPTIONAL] same as setting WHICH_MODEL = "phi" above and re-running send.
# In demo this only changes the tag in the canned reply. In live it hits Phi-4 Mini.
cmp_model = lab.endpoint_config()["comparison_model"]
cmp = lab.chat(messages, model=cmp_model)
print(cmp["model"], cmp["latency_ms"], "ms")
print(cmp["text"][:500])


Phi is the comparison model (open-weight, Azure-shaped). Same prompt, possibly different URL (`phi_url`). After lunch you will score both. Optional now so the room does not wait.

## Write-up (everyone)

1. Name the five parts of the stack you just used.
2. Both bank sentences end with the same words. What in the earlier words changed the continuation, and how would you explain that with the spotlight picture?
3. Under the banking-only instruction, did each reply do what the instruction asked? What changed in the model's weights between the two runs?
4. In the system-vs-user exercise, the **user** question stayed the same. What changed in the reply when you switched constitution? Which of the three would you ship, and why?
5. Which `chosen_id` did you send first, and which ministry EOI does that story stand in for? In one sentence: what did the **system** prompt forbid, and what was the **user** turn asking?
6. Did the reply try to **do** something (reset, hire, list serials, invent a circular) or **say** something? Which should it have done, given WATCH FOR?
7. What was `mode` on the result? If it was `demo` or `demo-fallback`, what would have to be true for it to be `live`?

If you loaded a second ministry id, note whether the same mechanic (constitution + this turn) still held.

Download this notebook at 14:55 with the rest. Overnight: you do **not** need to re-start vLLM to finish the write-up.
